# Retail Sales — Exploratory Data Analysis

This notebook validates and analyzes an educational retail dataset. It requires the cleaned, appropriately licensed input CSVs in `data_clean/`. Financial totals exclude two transactions with unknown quantities.

## 1. Load CSVs

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data_clean")
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Create data_clean/ with authorized cleaned countries.csv, events.csv and products.csv files.")

def load_csv(name):
    return pd.read_csv(DATA_DIR / name, keep_default_na=False, na_values=[""])

countries = load_csv("countries.csv")
events = load_csv("events.csv")
products = load_csv("products.csv")
print({name: len(df) for name, df in [("countries", countries), ("events", events), ("products", products)]})

## 2. Validate and type-check

Missing `units_sold` values remain unknown; do not replace them with zero.

In [ ]:
for col in ("units_sold", "unit_price", "unit_cost"):
    events[col] = pd.to_numeric(events[col], errors="coerce")
for col in ("order_date", "ship_date"):
    events[col] = pd.to_datetime(events[col], errors="raise")

print("Missing event fields:")
print(events.isna().sum()[lambda s: s > 0].to_string())
print("Sales channels:", events.sales_channel.value_counts().to_dict())
assert set(events.sales_channel) == {"ONLINE", "OFFLINE"}
assert events.units_sold.isna().sum() == 2
assert events.country_code.isna().sum() == 82
assert not products.product_id.duplicated().any()
assert not countries.country_code.duplicated().any()
assert not events.order_id.duplicated().any()

## 3. Join records safely

Many-to-one validation catches duplicated lookup keys.

In [ ]:
sales = events.merge(products, on="product_id", how="left", validate="many_to_one", indicator="product_merge")
sales = sales.merge(countries, on="country_code", how="left", validate="many_to_one", indicator="country_merge")
assert len(sales) == len(events)
sales["geography"] = sales["name"].fillna("UNKNOWN")
print("Product joins:", sales.product_merge.value_counts().to_dict())
print("Country joins:", sales.country_merge.value_counts().to_dict())

## 4. Financial and operational metrics

The two transactions with missing quantities have unknown monetary contributions.

In [ ]:
sales["revenue"] = sales.units_sold * sales.unit_price
sales["cost"] = sales.units_sold * sales.unit_cost
sales["profit"] = sales.revenue - sales.cost
sales["shipping_days"] = (sales.ship_date - sales.order_date).dt.days

revenue = sales.revenue.sum()
profit = sales.profit.sum()
margin = profit / revenue
print(f"Orders: {sales.order_id.nunique():,}")
print(f"Recorded revenue: ${revenue:,.2f}")
print(f"Recorded profit: ${profit:,.2f}")
print(f"Recorded gross margin: {margin:.2%}")
print(f"Missing quantity records: {sales.units_sold.isna().sum()}")
print(f"Mean shipping days: {sales.shipping_days.mean():.2f}")
print(f"Identified countries: {sales.name.nunique()}")

assert round(revenue, 2) == 1702129408.21
assert round(profit, 2) == 501434459.00

## 5. Compare categories and sales channels

In [ ]:
channel_summary = sales.groupby("sales_channel", dropna=False)[["revenue", "profit"]].sum().sort_values("profit", ascending=False)
category_summary = sales.groupby("item_type", dropna=False)[["revenue", "profit"]].sum().sort_values("profit", ascending=False)
print("Channel summary:")
print(channel_summary.round(2).to_string())
print("\nTop six categories:")
print(category_summary.head(6).round(2).to_string())
print("\nTop-three profit share:", f"{category_summary.profit.head(3).sum()/profit:.1%}")

## 6. Visualize

In [ ]:
import matplotlib.pyplot as plt
ax = (category_summary["profit"].head(8) / 1e6).sort_values().plot(kind="barh", figsize=(8, 5), title="Profit by product category (USD millions)")
ax.set_xlabel("Profit (USD millions)")
ax.set_ylabel("Product category")
plt.tight_layout()
plt.show()

ax = (channel_summary[["revenue", "profit"]] / 1e6).plot(kind="bar", figsize=(7, 4), title="Sales channel performance (USD millions)", rot=0)
ax.set_ylabel("USD millions")
plt.tight_layout()
plt.show()

## 7. Interpretation and limitations

- Top categories by recorded profit: Cosmetics, Office Supplies, and Household (~48.6% combined).
- Revenue/profit exclude two transactions with unknown quantity.
- 82 transactions lack country codes; do not assign geography by guesswork.
- Results describe a practice dataset, not audited business performance.
- These are descriptive patterns, not proven causes.